In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

Imports

In [4]:
!pip install langchain_community langchain_core langchain_text_splitters langchain_huggingface pytesseract pdf2image chromadb groq pymupdf -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 33.7 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 32.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 38.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 36.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 13.2 MB/s eta 0:

In [8]:
!apt-get update -q && apt-get install -y poppler-utils -q

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,495 kB]
Get:12 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-upd

 PDF padhna (OCR/Extraction):

In [9]:
import os
from pdf2image import convert_from_path
import pytesseract
from langchain_core.documents import Document

source_directory = "/kaggle/input/datasets/hadikp/resume-data-pdf/Resumes PDF/Accountant resumes"

# Saare PDF paths collect karo
pdf_paths = []
for root, dirs, files in os.walk(source_directory):
    for file in files:
        if file.lower().endswith('.pdf'):
            pdf_paths.append(os.path.join(root, file))

print(f"Total CVs found: {len(pdf_paths)}")

# Har PDF ka text nikalo using OCR
raw_documents = []
for path in pdf_paths:
    try:
        images = convert_from_path(path)
        file_text = ""
        for img in images:
            file_text += pytesseract.image_to_string(img) + "\n"
        if file_text.strip():
            raw_documents.append(Document(
                page_content=file_text,
                metadata={"source": path, "filename": os.path.basename(path)}
            ))
    except Exception as e:
        print(f"Skipped: {os.path.basename(path)} — {str(e)}")

print(f"Successfully extracted text from {len(raw_documents)} CVs")

Total CVs found: 67
Successfully extracted text from 67 CVs


Embeddings + ChromaDB

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Har CV ko ek hi chunk mein rakho (4000 chars kaafi hai ek resume ke liye)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=4000, chunk_overlap=250)
docs = text_splitter.split_documents(raw_documents)
print(f"Total chunks created: {len(docs)}")

# Embedding model — yeh text ko numbers mein convert karta hai
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# ChromaDB mein save karo
DB_DIR = "/kaggle/working/chroma_accountant"
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=DB_DIR
)
print(f"✅ Database ready! Total chunks stored: {vector_db._collection.count()}")

/tmp/ipykernel_58/3454337345.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


Total chunks created: 68


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Database ready! Total chunks stored: 68


Search + Results

In [13]:
import os
import time
import shutil
from groq import Groq
from IPython.display import FileLink, display

client = Groq(api_key="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")

def get_llm_summary(resume_text):
    prompt = (
        "You are a recruiter. Read this resume and write EXACTLY 2-3 crisp lines "
        "about this candidate — their role and top skills. "
        "Only mention years of experience if EXPLICITLY written as a number. "
        "If multiple experience durations are mentioned, always state the TOTAL/OVERALL experience first."
        "No bullet points. No intro line. Start with job title or candidate name.\n\n"
        + resume_text[:4000]
    )
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Summary failed: {str(e)}"

def make_downloadable(file_path, file_name):
    output_path = f"/kaggle/working/{file_name}"
    if not os.path.exists(output_path):
        shutil.copy2(file_path, output_path)
    return output_path

# Search karo
user_prompt = "Accountant with experience in GST, Tally, and Tax filing"
results = vector_db.similarity_search_with_score(user_prompt, k=20)

print(f"🎯 Top 3 Matches for: '{user_prompt}'\n")
print("=" * 60)

seen = set()
count = 0

for doc, score in results:
    fname = doc.metadata.get("filename", "unknown.pdf")
    fpath = doc.metadata.get("source", "")

    if fname in seen:
        continue
    seen.add(fname)
    count += 1

    percentage = round((1 - score) * 100, 2)

    print(f"\n#{count} 📄 {fname}")
    print(f"🔥 Match Score: {percentage}%")
    print(f"\n✨ Summary:")
    summary = get_llm_summary(doc.page_content)
    print(f"   {summary}")

    print(f"\n📥 Download:")
    working_path = make_downloadable(fpath, fname)
    display(FileLink(working_path))

    print("\n" + "=" * 60)
    time.sleep(1)

    if count == 3:
        break

🎯 Top 3 Matches for: 'Accountant with experience in GST, Tally, and Tax filing'


#1 📄 Image_72.pdf
🔥 Match Score: 17.5%

✨ Summary:
   Accountant with overall experience in Finance and Accounts, skilled in accounts payable process, invoice processing, and vendor master data maintenance. Top skills include processing non-PO and service PO invoices, validating freight invoices, and maintaining financial controls. Committed and dedicated professional with 2 years of experience in accounts payable.

📥 Download:


/kaggle/working/Image_72.pdf



#2 📄 Image_80.pdf
🔥 Match Score: 17.5%

✨ Summary:
   Veera Venkata Satyanarayana Mullapudi is an Accountant with expertise in financial accounting, asset management, and accounting packages like SAP-FICO and Tally ERP. His top skills include financial statement preparation, payroll management, and bank reconciliation. He is proficient in MS Office and SAP skills.

📥 Download:


/kaggle/working/Image_80.pdf



#3 📄 Image_76.pdf
🔥 Match Score: 16.73%

✨ Summary:
   SAIKRISHNA.P is an experienced Accountant with 2 years of experience, skilled in Tally, Wings, SAP FI-CO, and account management. He possesses expertise in creating and distributing AP and AR reports, with strong proficiency in MS Office and excellent communication skills.

📥 Download:


/kaggle/working/Image_76.pdf